# 08_transformer_device

对应 `index.md` 第 8 阶段：注意力 / Transformer 组件、device 迁移、常见调试坑。

笔记本在 `codes/pytorch/08_transformer_device/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "08_transformer_device" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "pytorch" / "08_transformer_device"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/pytorch/08_transformer_device')

## 1. 选 device

写一段代码：若 `torch.cuda.is_available()` 为真则用 `cuda`，否则用 `cpu`。

得到 `device` 后，打印它。


In [2]:
import torch

# 作答

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)
#评阅
# 对。按题意在 cuda/cpu 间选择。写成 torch.device(...) 也常见，字符串同样能用。

#参考答案
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(device)

/Users/keyficller/Documents/AEFS-Notes/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


cpu


## 2. 张量与模型迁到同一设备

已有 `net`、`x` 和 `device`。把模型和输入都迁到 `device`，再前向一次。

打印输出张量的 `device`。


In [5]:
import torch
from torch import nn

device = torch.device("cpu")
net = nn.Linear(4, 2)
x = torch.randn(3, 4)

# 作答

net = net.to(device)
x = x.to(device)

print(net(x).device)
#评阅
# 对。模型与输入都 .to(device) 后再前向。

#参考答案
# net = net.to(device)
# y = net(x.to(device))
# print(y.device)

cpu


## 3. MultiheadAttention 一次

用 `nn.MultiheadAttention`：`embed_dim=8`，`num_heads=2`，`batch_first=True`。

对已有的 `x`（当作 query=key=value）做自注意力，打印输出 `attn_out` 的 `shape`。


In [9]:
import torch
from torch import nn

x = torch.randn(2, 5, 8)

# 作答

mha = nn.MultiheadAttention(
    embed_dim = 8,
    num_heads=2,
    batch_first=True
)

attn_out, score = mha(x, x, x)

print(attn_out.shape)
print(score.shape)
#评阅
# 对。自注意力 q=k=v=x，attn_out shape [2, 5, 8]。多打 score 无妨。

#参考答案
# attn_out, _ = nn.MultiheadAttention(8, 2, batch_first=True)(x, x, x)
# print(attn_out.shape)

torch.Size([2, 5, 8])
torch.Size([2, 5, 5])


## 4. TransformerEncoderLayer

用 `nn.TransformerEncoderLayer`：`d_model=8`，`nhead=2`，`batch_first=True`，其余默认。

对 `x` 前向，打印输出的 `shape`（应与输入同形）。


In [10]:
import torch
from torch import nn

x = torch.randn(2, 6, 8)

# 作答

encode_layer = nn.TransformerEncoderLayer(
    d_model = 8,
    nhead = 2,
    batch_first=True
)

out = encode_layer(x)

print(out.shape)
print(x.shape)
#评阅
# 对。输出与输入同形 [2, 6, 8]。

#参考答案
# layer = nn.TransformerEncoderLayer(d_model=8, nhead=2, batch_first=True)
# print(layer(x).shape)

torch.Size([2, 6, 8])
torch.Size([2, 6, 8])


## 5. 对齐设备后再算

`net` 默认在 CPU；`x` 已放到当前可用的 `device`（cuda / mps / cpu）。

先打印：`next(net.parameters()).device` 与 `x.device` 是否相等。

再把 `net` 迁到 `x.device`，前向一次，打印输出的 `shape`。


In [13]:
import torch
from torch import nn

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

net = nn.Linear(3, 1)
x = torch.randn(2, 3).to(device)

# 作答

print(next(net.parameters()).device == x.device)

net = net.to(x.device)

print(next(net.parameters()).device == x.device)
print(net(x).shape)
#评阅
# 对。先比较 device（你这儿是 False，说明 x 在加速器），再 .to 对齐后前向，shape [2, 1]。

#参考答案
# print(next(net.parameters()).device == x.device)
# net = net.to(x.device)
# print(net(x).shape)


False
True
torch.Size([2, 1])
